# 11. VLM 오류 분석(공식 Validation, 선정 모델 SEL-A-s43)

공식 안내문의 "모델 평가 및 VLM 기반 오류 분석" 단계입니다. **VLM의 설명은 오류 원인을 확정하는 근거가 아니라, 확인할 가설을 만드는 재료**로만 씁니다.

- **대상 55장**: `src/vlm_select.py`의 규칙으로 이미지를 보기 전에 골랐습니다.
  - 오분류 40장: 클립당 1장, 확신도 높은 오답부터, 카메라를 돌아가며
  - 대조용 정답 15장
  - Test는 쓰지 않습니다.
- **호출**: 모델은 `gpt-6-luna`, 사진마다 두 번 호출합니다. 결과 원문은 `experiments/VLM/results.jsonl`에 있습니다.
  1. **blind**: 정답을 알려 주지 않고 12개 운동 중 무엇으로 보이는지 묻습니다.
  2. **explain**: 정답과 모델 예측을 알려 주고 안내문 항목을 묻습니다(시각 특징, 구분 단서, 가능한 오류 요인, 촬영 조건, 판단 어려운 점).
- 이 노트북은 API를 다시 부르지 않고, 저장된 응답만 읽습니다.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from common import DATA_RAW, FIGURES_SAMPLES, class_names, set_korean_font

print("font:", set_korean_font())
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
NAMES = class_names()
V = ROOT / "experiments" / "VLM"
cand = pd.read_csv(V / "candidates.csv")
res = [json.loads(l) for l in (V / "results.jsonl").read_text(encoding="utf-8").splitlines()]
ok = [r for r in res if r.get("ok")]
print(f"호출 {len(res)}회, 성공 {len(ok)}회, 실패 {len(res) - len(ok)}회")
tin, tout = sum(r["input_tokens"] for r in ok), sum(r["output_tokens"] for r in ok)
print(f"토큰 입력 {tin:,} / 출력 {tout:,} → 추정 비용 ${tin / 1e6 * 0.05 + tout / 1e6 * 0.25:.4f}(gpt-6-luna 공식 단가, 100만 토큰당 $0.05 / $0.25)")
B = {r["image_id"]: r["answer"] for r in ok if r["stage"] == "blind"}
E = {r["image_id"]: r["answer"] for r in ok if r["stage"] == "explain"}
d = cand.copy()
d["vlm_guess"] = d.image_id.map(lambda i: B.get(i, {}).get("guess_id"))
d["vlm_conf"] = d.image_id.map(lambda i: B.get(i, {}).get("confidence"))
d["vlm_second"] = d.image_id.map(lambda i: B.get(i, {}).get("second_guess_id"))
d["vlm_correct"] = d.vlm_guess == d.target_id
d["vlm_top2"] = d.vlm_correct | (d.vlm_second == d.target_id)
d["vlm_eq_model"] = d.vlm_guess == d.pred
d.groupby("kind")[["vlm_correct", "vlm_top2", "vlm_eq_model"]].mean().round(3).rename(
    columns={"vlm_correct": "VLM blind 정답률", "vlm_top2": "VLM 상위 2개 안에 정답", "vlm_eq_model": "VLM 답 = 모델 예측"})

font: Malgun Gothic
호출 110회, 성공 110회, 실패 0회
토큰 입력 49,183 / 출력 90,407 → 추정 비용 $0.0251(gpt-6-luna 공식 단가, 100만 토큰당 $0.05 / $0.25)


,VLM blind 정답률,VLM 상위 2개 안에 정답,VLM 답 = 모델 예측
kind,,,
오분류,0.525,0.625,0.225
정답,0.667,0.800,0.667


## 1. VLM도 같은 사진을 헷갈리는가(blind)

In [2]:
g = d[d.target_id.isin([2, 7, 9])]
print("바벨·덤벨 묶음(정답 2·7·9): 행 = 정답, 열 = VLM blind 답")
pd.crosstab(g.true_name, g.vlm_guess.map(lambda x: NAMES.get(x, "응답 없음") if pd.notna(x) else "응답 없음"), margins=True)

바벨·덤벨 묶음(정답 2·7·9): 행 = 정답, 열 = VLM blind 답


vlm_guess,Y - Exercise,덤벨 벤트오버 로우,딥스,바벨 데드리프트,바벨 로우,케이블 크런치,All
true_name,,,,,,,
덤벨 벤트오버 로우,2,4,0,3,1,0,10
바벨 데드리프트,4,0,0,6,0,0,10
바벨 로우,1,1,1,5,1,1,10
All,7,5,1,14,2,1,30


In [3]:
rows = []
for kind in ("오분류", "정답"):
    s = d[d.kind == kind]
    rows.append({"구분": f"모델 {kind}", "장수": len(s), "VLM 정답": int(s.vlm_correct.sum()),
                 "VLM이 모델과 같은 오답": int(((s.vlm_guess == s.pred) & (s.pred != s.target_id)).sum()),
                 "VLM이 다른 오답": int(((s.vlm_guess != s.pred) & ~s.vlm_correct).sum())})
pd.DataFrame(rows)

,구분,장수,VLM 정답,VLM이 모델과 같은 오답,VLM이 다른 오답
0,모델 오분류,40,21,9,10
1,모델 정답,15,10,0,5


## 2. 안내문 형식의 기록 표(오분류 40장)

In [4]:
def j(x):
    return " / ".join(x) if isinstance(x, list) else (x or "")
tab = d[d.kind == "오분류"].assign(
    예측결과="Incorrect",
    VLM_blind=lambda t: t.vlm_guess.map(lambda x: NAMES.get(x, "-") if pd.notna(x) else "-"),
    주요특징=lambda t: t.image_id.map(lambda i: j(E.get(i, {}).get("visual_features"))),
    구분단서=lambda t: t.image_id.map(lambda i: j(E.get(i, {}).get("distinguishing_cues"))),
    오류요인_가설=lambda t: t.image_id.map(lambda i: j(E.get(i, {}).get("possible_error_factors"))),
    동작단계=lambda t: t.image_id.map(lambda i: E.get(i, {}).get("motion_phase", "")),
    추가확인=lambda t: t.image_id.map(lambda i: j(E.get(i, {}).get("follow_up"))))
tab[["image_id", "true_name", "pred_name", "camera", "conf", "예측결과", "VLM_blind", "주요특징", "구분단서", "오류요인_가설", "동작단계", "추가확인"]]

,image_id,true_name,pred_name,camera,conf,예측결과,VLM_blind,주요특징,구분단서,오류요인_가설,동작단계,추가확인
0,fitness_d13c227d1122f75ecd8689b4,덤벨 벤트오버 로우,바벨 데드리프트,B,0.999630,Incorrect,덤벨 벤트오버 로우,"사람이 화면 중앙에 비교적 작게 보이며, 무릎을 약간 굽히고 상체를 앞으로 기울인 자세입니다. / 팔은 아래쪽으로 내려와 있고 손 부근에 작은 어두운 물체가 보이는 듯하지만, 사진 해상도상 물체의 형태는 분명...",덤벨 벤트오버 로우라면 팔꿈치를 뒤로 당기는 동작이나 팔이 아래로 늘어진 자세가 단서가 될 수 있습니다. 이 사진에서는 팔이 내려와 있지만 팔꿈치를 당기는 동작인지는 뚜렷하지 않습니다. / 바벨 데드리프트의 ...,상체를 앞으로 기울이고 무릎을 굽힌 자세가 두 운동에서 모두 나타날 수 있어 자세만으로 구별하기 어려웠을 가능성이 있습니다. / 사람과 손에 든 물체가 화면에서 작아 운동 기구의 형태가 충분히 드러나지 않았을...,"알 수 없음. 상체를 기울이고 팔을 내린 순간은 보이지만, 한 장의 사진만으로 동작의 시작·중간·끝을 특정하기 어렵습니다.",손과 운동 기구가 더 크게 보이는 원본 또는 확대 이미지를 확인합니다. / 연속 프레임에서 팔꿈치와 중량이 움직이는 방향을 확인합니다. / 동일 운동의 다른 시점 이미지와 함께 분류 성능을 비교합니다.
1,fitness_2403826389cae0182842d1dc,덤벨 벤트오버 로우,바벨 런지,E,0.990709,Incorrect,Y - Exercise,"사람이 화면 중앙에 등을 보인 채 서 있다. / 상체는 대체로 세워져 있고, 양팔은 몸통에서 옆으로 벌어져 팔꿈치가 굽은 듯 보인다. / 다리는 서로 가까이 놓여 있으며, 화면에서 뚜렷한 앞뒤 보폭은 보이지 ...",덤벨 벤트오버 로우의 단서인 상체를 앞으로 숙인 자세와 팔꿈치를 몸통 쪽으로 당기는 동작은 사진에서 뚜렷하지 않다. / 바벨 런지의 단서인 바벨을 어깨에 얹은 모습과 앞뒤로 벌린 다리 자세도 뚜렷하게 보이지 ...,사람이 화면에서 작게 보여 손과 운동 기구의 형태가 구분되지 않았을 가능성이 있다. / 동작 중 한 순간만 담겨 있어 운동의 전형적인 자세가 나타나지 않았을 가능성이 있다. / 등을 보이는 시점과 팔을 옆으로...,"알 수 없음. 사진에는 상체가 세워진 자세가 보이지만, 운동의 시작·중간·끝 단계인지 판단할 만큼 동작의 전후 정보가 없다.","손과 어깨 주변을 확대해 운동 기구가 있는지 확인한다. / 연속 프레임이나 영상에서 상체 기울기, 팔의 당김 동작, 다리의 보폭 변화를 확인한다. / 사람이 더 크게 보이도록 자른 이미지나 더 높은 해상도의 ..."
2,fitness_5f9ebcdef00ef90de62850f2,덤벨 벤트오버 로우,바벨 데드리프트,D,0.763683,Incorrect,바벨 데드리프트,한 사람이 화면 중앙에서 상체를 앞으로 숙이고 서 있으며 무릎은 약간 굽혀져 있다. / 양팔은 아래로 내려가 있고 손은 무릎 부근에 있다. 손에 작은 무게기구가 있는 듯하지만 형태는 선명하지 않다. / 운동 ...,"덤벨 벤트오버 로우에 해당할 수 있는 숙인 자세와 손의 무게기구는 보이지만, 팔꿈치를 굽혀 무게를 몸통 쪽으로 당기는 모습은 뚜렷하지 않다. / 바벨 데드리프트에서 기대할 수 있는 양손으로 잡은 바벨은 사진에...",상체를 숙이고 무게를 든 자세가 두 운동에서 모두 나타나 시각적으로 비슷하게 보였을 가능성이 있다. / 팔과 무게기구의 형태가 작고 선명하지 않아 덤벨인지 다른 기구인지 식별하기 어려웠을 수 있다. / 로우 ...,"알 수 없음. 상체를 숙이고 팔을 아래로 내린 순간은 확인되지만, 한 장의 사진만으로 시작·중간·끝 단계나 동작 방향을 판별하기 어렵다.",손과 무게기구 주변을 확대해 기구의 형태와 양손의 파지 상태를 확인한다. / 연속 프레임에서 팔꿈치가 굽혀지며 무게가 몸통 쪽으로 올라가는지 확인한다. / 같은 장면의 더 높은 해상도 이미지나 다른 촬영 각도...
3,fitness_f9e8801d0e055c352a7b42ba,덤벨 벤트오버 로우,바벨 데드리프트,B,0.996532,Incorrect,덤벨 벤트오버 로우,"헬스장 바닥과 여러 운동기구가 넓게 보이고, 사람은 화면 중앙에 전신이 작게 잡혀 있다. / 사람은 서서 상체를 약간 앞으로 기울인 듯하며, 양팔은 아래로 내려가 있다. / 손에 든 물체가 보이지만 사진 크기...","덤벨 벤트오버 로우라면 보통 엉덩이를 접어 상체를 더 숙이고 팔꿈치를 뒤로 당기는 동작이 단서가 되지만, 이 사진에서는 뚜렷한 당김 동작이 확인되지 않는다. / 바벨 데드리프트라면 양손으로 잡은 바벨과 다리 ...","사람과 손의 무게가 작게 보여 도구의 형태와 팔꿈치 움직임이 불분명할 수 있다. / 두 운동 모두 상체를 앞으로 기울이고 손에 무게를 드는 자세가 나타날 수 있어, 이 순간의 자세만으로 비슷하게 보였을 가능성...","알 수 없음. 팔이 아래로 내려가 있고 상체가 약간 기울어 보이지만, 한 장의 정지 사진만으로 로우의 당김 전후나 동작 단계를 특정하기 어렵다.","손과 무게가 더 크게 보이는 고해상도 사진이나 연속 프레임을 확인한다. / 동작 중 팔꿈치가 뒤로 당겨지는지, 무게가 다리 앞에서 위아래로 이동하는지 확인한다. / 덤벨 벤트오버 로우와 바벨 데드리프트의 유사..."
4,fitness_164a253c358da44e342ddb98,덤벨 벤트오버 로우,바벨 데드리프트,E,0.778033,Incorrect,바벨 데드리프트,"한 사람이 화면 중앙에서 상체를 앞으로 숙이고 엉덩이를 뒤로 뺀 자세를 취하고 있다. / 무릎은 약간 굽혀져 있고, 팔은 아래쪽으로 뻗어 손이 무릎 부근에 있다. / 손에 든 물체가 있는 듯 보이지만, 물체의...",덤벨 벤트오버 로우에서 단서가 될 수 있는 양손의 덤벨은 이 사진에서 분명히 식별되지 않는다. / 바벨 데드리프트에서 단서가 될 수 있는 긴 바벨도 뚜렷하게 보이지 않는다. / 팔꿈치를 굽혀 무게를 몸통 쪽으...,"두 운동 모두 상체를 숙이고 무게를 아래로 두는 자세가 나타날 수 있어, 이 한 순간의 자세가 비슷하게 보였을 가능성이 있다. / 손의 무게 도구가 작고 불분명해 덤벨인지 바벨인지 구별하기 어려웠을 수 있다....","알 수 없음. 상체를 숙이고 팔을 아래로 둔 순간은 보이지만, 시작·중간·끝 중 어느 단계인지는 한 장만으로 판단하기 어렵다.",손과 운동 도구가 더 크게 보이는 고해상도 이미지를 확인한다. / 연속 프레임이나 영상에서 팔꿈치가 굽혀지며 무게를 몸통 쪽으로 당기는지 확인한다. / 운동 도구가 덤벨인지 바벨인지 다른 각도에서 확인한다.
5,fitness_703278fa1b8bba415b118990,덤벨 벤트오버 로우,바벨 데드리프트,B,0.992379,Incorrect,Y - Exercise,"한 사람이 화면 중앙에서 전신이 보이도록 서 있다. / 상체는 대체로 세워져 있고 팔은 몸 옆으로 내려와 있다. / 손 주변에 작은 무게추처럼 보이는 물체가 있으나, 사진 해상도만으로 형태를 확실히 구분하기 ...",덤벨 벤트오버 로우에서 흔히 보이는 상체를 숙인 자세와 팔꿈치를 뒤로 당기는 동작은 이 사진에서 뚜렷하게 보이지 않는다. / 바벨 데드리프트에서 흔히 보이는 상체·엉덩이의 힌지 자세와 다리 앞쪽의 바벨도 뚜렷...,사진이 운동 동작의 전형적인 자세가 아니라 시작 또는 종료 시점일 가능성이 있다. / 상체가 세워진 순간이라면 두 운동을 가르는 자세 단서가 약할 수 있다. / 무게추와 손의 세부 형태가 작게 보여 기구 종류...,"알 수 없음 — 현재 사진에서는 사람이 거의 똑바로 서 있어, 운동 전후의 순간인지 동작 중 잠시 자세를 바꾼 것인지 구분하기 어렵다.","연속 프레임이나 운동 영상에서 상체 각도와 팔의 움직임을 확인한다. / 손과 중량 기구가 더 크게 보이는 원본 또는 고해상도 이미지를 확인한다. / 모델의 해당 예측에 대한 근거를 살펴보고, 정답 클래스와 예..."
6,fitness_b0c8e772012657a6f5586734,덤벨 벤트오버 로우,바

## 3. VLM이 든 오류 요인 모으기(키워드 집계, 가설 후보)

In [5]:
# 단순 포함 검사라 거칠다. "작"은 "동작"에, "뒤"는 "엉덩이를 뒤로"에도 걸려서 더 긴 표현만 쓴다.
KEYS = {"상체 숙임·기울기": ["상체", "숙", "기울"], "기구 식별(바벨/덤벨)": ["바벨", "덤벨", "원판", "막대", "기구"],
        "손·팔·그립": ["손", "그립", "팔"], "한 순간·동작 단계": ["시작", "중간", "단계", "순간", "정지", "움직임"],
        "카메라 시점": ["정면", "측면", "뒤쪽에서", "뒤에서", "후면", "등을 보", "시점", "촬영 각도"],
        "작은 사람·해상도": ["작게", "작은", "작고", "해상도", "흐릿", "선명하지", "멀리", "픽셀"],
        "가림": ["가림", "가려", "겹쳐"], "배경·넓은 공간": ["배경", "넓은", "장소"], "조명·밝기": ["조명", "밝기", "어둡", "역광"]}
def count(field, kind):
    texts = [j(E.get(i, {}).get(field)) for i in d[d.kind == kind].image_id]
    return {k: sum(any(w in t for w in ws) for t in texts) for k, ws in KEYS.items()}
pd.DataFrame({"오분류: 오류 요인": count("possible_error_factors", "오분류"), "오분류: 판단 어려운 점": count("hard_to_judge", "오분류"),
              "정답: 판단 어려운 점": count("hard_to_judge", "정답")}).sort_values("오분류: 오류 요인", ascending=False)

,오분류: 오류 요인,오분류: 판단 어려운 점,정답: 판단 어려운 점
한 순간·동작 단계,36,23,8
작은 사람·해상도,33,3,1
손·팔·그립,30,30,10
기구 식별(바벨/덤벨),30,31,7
상체 숙임·기울기,26,13,2
가림,9,2,1
카메라 시점,8,2,5
배경·넓은 공간,6,0,0
조명·밝기,0,0,0


## 4. 대표 사례 보기(원본 사진, 공개 저장소에서는 그림 출력을 지움)

In [6]:
show = d[(d.kind == "오분류") & d.target_id.isin([2, 7, 9])].groupby("target_id").head(3)
fig, axes = plt.subplots(3, 3, figsize=(18, 11))
for ax, r in zip(axes.ravel(), show.itertuples()):
    ax.imshow(Image.open(DATA_RAW / r.image_path)); ax.axis("off")
    vg = NAMES.get(r.vlm_guess, "-") if pd.notna(r.vlm_guess) else "-"
    ax.set_title(f"정답 {r.true_name} → 모델 {r.pred_name}({r.conf:.2f})\ncam{r.camera} | VLM blind: {vg}", fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES_SAMPLES / "vlm_cases_279.jpg", dpi=70); plt.show()

[원본 사진 그림 출력은 지웠습니다. 로컬 파일 figures/samples/에 저장되어 있고, 공개 저장소에는 넣지 않습니다(AI Hub 이용 조건).]


## 5. 해석(2026-11-26, 저장된 응답 110개 기준)

### 숫자로 본 결과
- 호출 110회가 모두 성공했고, 실패는 0회였습니다. 비용은 약 $0.025(추정, 공식 단가 기준)로, 상한 $3의 1% 안쪽입니다.
- **VLM도 같은 사진을 헷갈렸습니다.** 바벨·덤벨 묶음(정답 2·7·9) 30장에서 VLM의 blind 정답은 11장(37%)뿐입니다.
  - 정답별로 보면 덤벨 로우 4/10, 데드리프트 6/10, **바벨 로우 1/10**입니다.
  - VLM은 30장 중 14장을 "데드리프트"라고 답했습니다. 선 자세 바벨·덤벨 사진을 데드리프트로 몰아 읽는 경향이 모델과 같습니다.
- 묶음 밖(맨몸·기구) 25장에서는 VLM이 20장(80%)을 맞혔습니다. 모델이 틀린 16장 중에서도 12장을 맞혔습니다. 어려움이 2·7·9에 몰려 있다는 기존 오류 분석(E06)과 같은 그림입니다.
- VLM이 스스로 매긴 확신도는 모델 오분류 40장 중 25장에서 "낮음"이었습니다.

### 이 결과를 어떻게 읽을 것인가
- VLM에는 원본 파일을 `detail: high`로 보냈습니다. API 안에서 줄일 수는 있지만 224보다는 훨씬 큽니다. 그래서 모델 입력보다 정보가 많은데도 바벨 로우를 거의 맞히지 못했습니다. 이것은 **"바벨 로우와 데드리프트는 사진 한 장으로는 원래 구분하기 어렵다"는 가설을 지지**합니다. 헬스 관점에서도 두 운동은 같은 힙 힌지 자세에서 출발하고, 차이는 "바벨이 어디로 움직이는가"입니다.
- VLM은 정답이 아닙니다. 범용 모델이 이 데이터의 12개 운동을 잘 아는 것도 아닙니다. 그래서 위 내용은 원인의 확정이 아니라 **검증할 가설**로만 씁니다.

### VLM이 든 오류 요인(키워드 집계, 오분류 40장)
- 집계는 단어가 들어 있는지만 보는 거친 방법입니다. 순위만 참고합니다.

| 요인 | 언급 장수 | 정답 사진의 "판단 어려운 점"에서 |
|---|---|---|
| 한 순간만 담김·동작 단계 | 36 | 8/15 |
| 사람이 작음·해상도 | 33 | 1/15 |
| 기구 식별(바벨/덤벨) | 30 | 7/15 |
| 손·팔·그립 | 30 | 10/15 |

### 개선 가설(다음 실험 후보)
1. **한 장의 정보 한계 → 클립 8프레임 확률 평균.** E06 진단에서 바벨 로우가 크게 나아졌습니다. 이를 정식 비교로 만듭니다. 이미지 단위와 클립 단위 Macro F1을 같은 Val·CV fold에서 비교합니다. 다만 이것은 "추론 방식"을 바꾸는 것이라, 이미지 단위 과제의 점수와 섞지 않고 따로 보고합니다.
2. **사람이 작음 → 사람을 크게 보이게 하는 전처리.** P1에서 가운데 자르기(ccrop)가 가장 좋았던 결과와 같은 방향입니다. CV1에서 ccrop이 fold를 바꿔도 유지되는지 먼저 확인합니다. 사람 검출로 자르는 방법은 수업 범위 밖이라 비용을 설명한 뒤에 정합니다.
3. **기구 식별 → 색·형태 단서를 지키는 쪽.** 흑백 입력에서 덤벨 로우가 크게 떨어진 결과(P1 gray, 0.83→0.51)와 맞습니다. 색을 지우는 전처리와 증강은 피합니다. Grad-CAM으로 모델이 손·기구 부위를 보는지 확인합니다.